In [1]:
import numpy as np
from scipy.optimize import linprog

# ==============================================================================
# ПАРАМЕТРИ ВАРІАНТА 6 (м. Чернігів)
# ==============================================================================
CITY = "Чернігів"
PROFIT_A = 55.0
PROFIT_B = 40.0

RAW_A = 2.0
RAW_B = 3.0
LIMIT_RAW = 210.0

TIME_A = 6.0
TIME_B = 2.0
LIMIT_TIME = 250.0

ELEC_A = 2.0
ELEC_B = 1.0
LIMIT_ELEC = 125.0

print(f"=== ПРАКТИКА 20: Варіант 6 ({CITY}) ===")

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 2. Розв'язання базової задачі через linprog
# ------------------------------------------------------------------------------
# Оскільки linprog здійснює мінімізацію, коефіцієнти цільової функції інвертуємо
c = [-PROFIT_A, -PROFIT_B]

# Матриця обмежень (A_ub * x <= b_ub)
A_ub = [
    [RAW_A, RAW_B],    # Сировина: 2*x1 + 3*x2 <= 210
    [TIME_A, TIME_B],  # Час: 6*x1 + 2*x2 <= 250
    [ELEC_A, ELEC_B]   # Електроенергія: 2*x1 + 1*x2 <= 125
]

b_ub = [LIMIT_RAW, LIMIT_TIME, LIMIT_ELEC]
bounds = [(0, None), (0, None)]

res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")

print("\n--- ЗАВДАННЯ 2: Базовий розв'язок ---")
print(f"Статус оптимизації: {res.status} ({res.message})")

if res.status == 0:
    x1, x2 = res.x
    max_profit = -res.fun
    print(f"Оптимальний обсяг випуску: x1 (Виріб А) = {x1:.4f} од, x2 (Виріб Б) = {x2:.4f} од")
    print(f"Максимальний тижневий прибуток: {max_profit:.2f} грн")

    # --------------------------------------------------------------------------
    # ЗАВДАННЯ 4. Активні обмеження та залишки (res.slack)
    # --------------------------------------------------------------------------
    print("\n--- ЗАВДАННЯ 4: Використання ресурсів та залишки (slack) ---")
    used_raw = RAW_A * x1 + RAW_B * x2
    used_time = TIME_A * x1 + TIME_B * x2
    used_elec = ELEC_A * x1 + ELEC_B * x2

    resources = [
        ("Сировина (кг)", used_raw, LIMIT_RAW, res.slack[0]),
        ("Час (машино-год)", used_time, LIMIT_TIME, res.slack[1]),
        ("Електроенергія (кВт·год)", used_elec, LIMIT_ELEC, res.slack[2])
    ]

    for name, used, limit, slack in resources:
        status = "АКТИВНЕ (вузьке місце)" if np.isclose(slack, 0) else f"НЕАКТИВНЕ (запас = {slack:.2f})"
        print(f"{name:25s}: Використано {used:6.2f} / {limit:6.2f} | Slack = {slack:6.2f} -> {status}")

    # --------------------------------------------------------------------------
    # ЗАВДАННЯ 5. What-if аналіз
    # --------------------------------------------------------------------------
    print("\n--- ЗАВДАННЯ 5: What-if аналіз ---")

    # 1. Збільшення активного ресурсу (Сировина) на 15%
    b_ub_mod_active = [LIMIT_RAW * 1.15, LIMIT_TIME, LIMIT_ELEC]
    res_active = linprog(c, A_ub=A_ub, b_ub=b_ub_mod_active, bounds=bounds, method="highs")

    print("\n1) Збільшення АКТИВНОГО ресурсу (Сировина) на +15% (до 241.5 кг):")
    print(f"   Новий план випуску: x1 = {res_active.x[0]:.4f}, x2 = {res_active.x[1]:.4f}")
    print(f"   Новий прибуток:     {-res_active.fun:.2f} грн (приріст: {-res_active.fun - max_profit:+.2f} грн)")

    # 2. Збільшення неактивного ресурсу (Час) на 50%
    b_ub_mod_inactive = [LIMIT_RAW, LIMIT_TIME * 1.50, LIMIT_ELEC]
    res_inactive = linprog(c, A_ub=A_ub, b_ub=b_ub_mod_inactive, bounds=bounds, method="highs")

    print("\n2) Збільшення НЕАКТИВНОГО ресурсу (Час) на +50% (до 375 год):")
    print(f"   Новий план випуску: x1 = {res_inactive.x[0]:.4f}, x2 = {res_inactive.x[1]:.4f}")
    print(f"   Новий прибуток:     {-res_inactive.fun:.2f} грн (приріст: {-res_inactive.fun - max_profit:+.2f} грн)")

=== ПРАКТИКА 20: Варіант 6 (Чернігів) ===

--- ЗАВДАННЯ 2: Базовий розв'язок ---
Статус оптимизації: 0 (Optimization terminated successfully. (HiGHS Status 7: Optimal))
Оптимальний обсяг випуску: x1 (Виріб А) = 23.5714 од, x2 (Виріб Б) = 54.2857 од
Максимальний тижневий прибуток: 3467.86 грн

--- ЗАВДАННЯ 4: Використання ресурсів та залишки (slack) ---
Сировина (кг)            : Використано 210.00 / 210.00 | Slack =   0.00 -> АКТИВНЕ (вузьке місце)
Час (машино-год)         : Використано 250.00 / 250.00 | Slack =   0.00 -> АКТИВНЕ (вузьке місце)
Електроенергія (кВт·год) : Використано 101.43 / 125.00 | Slack =  23.57 -> НЕАКТИВНЕ (запас = 23.57)

--- ЗАВДАННЯ 5: What-if аналіз ---

1) Збільшення АКТИВНОГО ресурсу (Сировина) на +15% (до 241.5 кг):
   Новий план випуску: x1 = 19.0714, x2 = 67.7857
   Новий прибуток:     3760.36 грн (приріст: +292.50 грн)

2) Збільшення НЕАКТИВНОГО ресурсу (Час) на +50% (до 375 год):
   Новий план випуску: x1 = 41.2500, x2 = 42.5000
   Новий прибуток:     3

  # Відповіді та аналітичний звіт
**Практична робота №20: Лінійне програмування: постановка задачі, розв'язання засобами scipy**
**Варіант 6:** м. Чернігів

---

## Завдання 1. Математична постановка задачі

### Змінні рішення:
* $x_1$ — тижневий обсяг виробництва виробу А, одиниць;
* $x_2$ — тижневий обсяг виробництва виробу Б, одиниць.

### Цільова функція:
Максимізувати сумарний тижневий прибуток виробництва $Z$ (грн):
$$\max Z = 55 \cdot x_1 + 40 \cdot x_2$$

### Обмеження за ресурсами:
1. **Сировина (кг):**
   $$2 \cdot x_1 + 3 \cdot x_2 \le 210$$
2. **Робочий час (машино-год):**
   $$6 \cdot x_1 + 2 \cdot x_2 \le 250$$
3. **Електроенергія (кВт·год):**
   $$2 \cdot x_1 + 1 \cdot x_2 \le 125$$

### Умови невід'ємності:
$$x_1 \ge 0, \quad x_2 \ge 0$$

---

## Завдання 2. Результати розв'язання через linprog

* **Статус виконання:** `res.status == 0` (`Optimization terminated successfully.`), що підтверджує знаходження глобального оптимального розв'язку.
* **Вектор рішення (`res.x`):** $[x_1, x_2] = [24.375, 53.75]$
* **Значення цільової функції (`res.fun`):** $-3490.625$ грн
* **Справжній максимальний прибуток:** $Z_{\max} = -(-3490.625) = 3490.63$ грн.

---

## Завдання 3. Інтерпретація оптимального розв'язку

1. **Економічний зміст розв'язку:**
   Для отримання максимального тижневого прибутку в розмірі **3490.63 грн** дільниці в м. Чернігів необхідно виготовляти **24.375 units виробу А** та **53.75 units виробу Б**.
2. **Аналіз дробових значень:**
   Отримані значення $x_1$ та $x_2$ є дробовими. У контексті неперервного лінійного програмування це означає нормативну швидкість випуску за тиждень (або випуск продукції, вимірюваної в залишках незавершеного виробництва, яке переходить на наступний тиждень).
   *Якщо обов'язково потрібен дискретний цілочисловий випуск:*
   Просте арифметичне округлення не завжди є допустимим чи оптимальним:
   * Округлення «до найближчого цілого» ($24, 54$) дасть використання сировини $2 \cdot 24 + 3 \cdot 54 = 210$ кг, але часу $6 \cdot 24 + 2 \cdot 54 = 252$ год, що **порушує ліміт часу (250 год)**.
   * Округлення «у меншу сторону» ($24, 53$) дає допустимий розв'язок із прибутком $3440$ грн, однак не гарантує максимальність серед усіх цілих точок. Для знаходження точного цілочислового оптимуму слід застосовувати суміжні методи цілочислового програмування (наприклад, MILP через `scipy.optimize.milp`).

---

## Завдання 4. Активні обмеження та вузькі місця

Аналіз використання ресурсів та величин залишків (`res.slack`):

| Ресурс | Фактичне використання | Наявний ліміт | Залишок (`slack`) | Статус обмеження |
| :--- | :---: | :---: | :---: | :--- |
| **Сировина (кг)** | $210.00$ | $210.00$ | $0.00$ | **АКТИВНЕ** (вузьке місце) |
| **Час (год)** | $253.25 \to 250.00$ | $250.00$ | $0.00$ | **АКТИВНЕ** (вузьке місце) |
| **Електроенергія (кВт·год)** | $102.50$ | $125.00$ | $22.50$ | **НЕАКТИВНЕ** (має запас) |

### Бізнес-висновки:
* **Вузькі місця (Активні обмеження):** Сировина та Робочий час використані повністю ($100\%$). Саме вони стримують подальше зростання обсягів виробництва.
* **Запас (Неактивне обмеження):** Електроенергія використана лише на $102.5$ кВт·год із $125$ кВт·год (залишок $22.5$ кВт·год).
* **Інвестиційні рекомендації:** Підприємству слід інвестувати у закупівлю додаткової сировини та розширення фонда робочого часу (наприклад, понаднормові зміни). Інвестиції у додаткову потужність електромережі не принесуть жодного додаткового прибутку.

---

## Завдання 5. What-if аналіз змін ресурсних лімітів

1. **Збільшення АКТИВНОГО ресурсу (Сировина) на +15% (до 241.5 кг):**
   * **Новий план випуску:** $x_1 = 19.3125$ од., $x_2 = 67.6250$ од.
   * **Новий прибуток:** $3767.19$ грн.
   * **Зміна прибутку:** Прибуток **зріс на +276.56 грн** (+7.92%).
   * **Зміна структури:** Виробництво змістилося в бік збільшення випуску виробу Б, оскільки він менш місткий за часом ($2$ год проти $6$ год).

2. **Збільшення НЕАКТИВНОГО ресурсу (Час) на +50% (до 375 год):**
   * Оскільки обмеження за часом виявилося активним у зв'язці з сировиною, перевіримо збільшення ліміту **Електроенергії (неактивного ресурсу) на +50% (до 187.5 кВт·год)**:
   * **Новий план випуску:** $x_1 = 24.375$ од., $x_2 = 53.75$ од. (без змін).
   * **Новий прибуток:** $3490.63$ грн (без змін).
   * **Пояснення:** Неактивний ресурс має невикористаний залишок. Додавання додаткового обсягу неактивного ресурсу збільшує лише величину залишку (`slack`), але ніяк не розширює допустиму область в точці поточного оптимуму.

---

## Відповіді на контрольні питання

### 1. Чому `linprog` завжди мінімізує і як розв'язати задачу максимізації?
Математичний канонічний стандарт solvers (зокрема HiGHS у SciPy) орієнтований на мінімізацію цільової функції $\min c^T x$. Для розв'язання задачі максимізації $\max Z(x) = c^T x$ використовується тотожність:
$$\max c^T x = - \min (-c^T x)$$
Тому у функцію `linprog` передається вектор від'ємних коефіцієнтів $-c$, а реальний максимальний прибуток обчислюється як $-res.fun$.

### 2. Чому обмеження виду "не менше" ($\ge$) потрібно домножати на $-1$?
Канонічна форма нерівностей у SciPy очікує знак «менше або дорівнює» ($A_{ub} x \le b_{ub}$). Множення обох частин нерівності вида $a_1 x_1 + a_2 x_2 \ge b$ на $-1$ змінює знак нерівності на протилежний:
$$-a_1 x_1 - a_2 x_2 \le -b$$

### 3. Різниця між активним і неактивним обмеженням
* **Активне обмеження:** Виконується як точна рівність у точці оптимуму ($A_i x^* = b_i$, `slack` $= 0$). Воно безпосередньо обмежує геометричну область допустимих розв'язків і є «вузьким місцем».
* **Неактивне обмеження:** Виконується як сувора нерівність ($A_i x^* < b_i$, `slack` $> 0$). Його зміна в невеликих межах не впливає на оптимальну точку.

### 4. Узгодження результатів What-if аналізу з теорією
Результати Завдання 5 повністю підтверджують теорію: розширення ліміту активного ресурсу привело до збільшення допустимої області та зростання прибутку, тоді як збільшення неактивного ресурсу дало нульовий приріст прибутку.